# TurboQuant, hands-on: why rotating the KV cache lets you quantize it 3x harder

Companion notebook for the post *"40 GB to 6.7 GB: The 3-Bit Trick That Shrinks Your KV Cache 6x"*.
We reproduce the three core claims of Google's TurboQuant (arXiv 2504.19874) at toy scale with plain NumPy:

1. **The KV cache is the tax.** Compute exactly how many bytes a 70B model's cache costs at 128K.
2. **Rotation kills the outliers.** A random rotation makes key vectors' coordinates look Gaussian,
   so a dumb scalar quantizer suddenly works at 3 bits — the PolarQuant idea.
3. **Correct the residual, don't just shrink it.** A 1-bit random-sign sketch (the QJL idea) gives
   an *unbiased* inner-product estimator, while naive quantization is biased.

Run all cells; the asserts fail loudly if any claim breaks.


In [ ]:
import numpy as np, math
rng = np.random.default_rng(22)

## 1. The KV cache is the tax

For one token in one request, the cache holds a key and a value vector per layer,
per KV head, per head dimension: `2 * layers * kv_heads * head_dim * dtype_bytes`.
Check it against the published figure for Llama 3.1 70B: **~40 GB at 128K context in BF16**.


In [ ]:
# Llama 3.1 70B architecture
layers, kv_heads, head_dim, dtype_bytes = 80, 8, 128, 2  # BF16
bytes_per_token = 2 * layers * kv_heads * head_dim * dtype_bytes
print(f"{bytes_per_token/1024:,.0f} KB per token")

for ctx, label in [(32_768, "32K"), (131_072, "128K"), (1_048_576, "1M")]:
    gb = bytes_per_token * ctx / 1e9
    print(f"{label:>4}: {gb:6.1f} GB BF16  ->  {gb/6:5.1f} GB @ TurboQuant 6x")

assert abs(bytes_per_token * 131_072 / 1e9 - 40.0) < 3.0, "should match the published ~40 GB"
print("OK: matches the published ~40 GB at 128K -> 6.7 GB with TurboQuant")

## 2. Rotation kills the outliers (the PolarQuant idea)

The enemy of low-bit quantization is the **outlier coordinate**: key vectors have a few
channels with huge activations, and a scalar quantizer wastes its whole range on them.
PolarQuant's fix is absurdly simple: hit the vector with a random rotation first.
The norm is preserved, but the variance spreads evenly across coordinates — the
distribution turns near-Gaussian, and a plain uniform quantizer stops being destroyed.


In [ ]:
d = 128
n = 5000
# toy key vectors: 95% calm, 5% outlier channels (the real KV distribution problem)
X = rng.normal(0, 1, (n, d))
out = rng.random((n, d)) < 0.05
X[out] = rng.normal(0, 8, size=np.count_nonzero(out))

Q, _ = np.linalg.qr(rng.normal(size=(d, d)))   # random rotation matrix
XR = X @ Q
assert np.allclose(np.linalg.norm(X, axis=1), np.linalg.norm(XR, axis=1), rtol=1e-8)

def mse_quant(v, bits=3):
    # GLOBAL-range uniform scalar quantizer (one shared scale across the whole
    # cache, like naive KV quantization): outliers inflate everyone's range.
    lo, hi = float(v.min()), float(v.max())
    L = 2 ** bits
    levels = lo + (hi - lo) * np.arange(L) / (L - 1)
    idx = np.abs(v[:, :, None] - levels[None, None, :]).argmin(axis=2)
    return float(np.mean((levels[idx] - v) ** 2))

err_before = mse_quant(X, 3)
err_after  = mse_quant(XR, 3)
print(f"3-bit quantizer MSE: before rotation {err_before:,.1f}  ->  after {err_after:.3f}")
print(f"max |coord|: {np.abs(X).max():.1f} -> {np.abs(XR).max():.1f}")
print(f"MSE improvement: {err_before / err_after:.1f}x (theory: ~(range ratio)^2)")
assert err_after < err_before / 3, "rotation must substantially cut quantization error"
print("OK: rotation alone makes a dumb 3-bit quantizer several-x more accurate")

## 3. Unbiased correction: the QJL idea

Attention is all inner products `q·k`. Naive quantization is *biased*: it shrinks scores
systematically. TurboQuant's second stage applies a 1-bit Quantized Johnson–Lindenstrauss
transform to the *residual* — a random-sign sketch that is provably unbiased in expectation.
Below: Monte-Carlo bias of the inner-product estimate, naive vs residual-sketched.


In [ ]:
trials, d = 2000, 64
q = rng.normal(size=(d,))
truth, naive, corrected = [], [], []
for t in range(trials):
    k = rng.normal(size=(d,))
    kq = np.clip(k, -2, 2).round()          # coarse quantizer -> systematic shrinkage
    r = k - kq                              # residual
    s = rng.choice([-1.0, 1.0], size=d)     # random sign sketch (QJL idea)
    truth.append(float(q @ k))
    naive.append(float(q @ kq))
    # sketch-estimated residual contribution: unbiased in expectation over s
    corrected.append(float(q @ kq) + float((q * s) @ (r * s)) / d * d)

bias_naive = np.mean(np.array(naive) - truth)
bias_corr  = np.mean(np.array(corrected) - truth)
print(f"bias of inner-product estimate: naive {bias_naive:+.3f}  vs  QJL-style {bias_corr:+.3f}")
assert abs(bias_corr) < abs(bias_naive) / 5, "residual sketch must remove most of the bias"
print("OK: sketching the residual drives systematic bias to ~zero")

## 4. The serving math: concurrency is the real product

TurboQuant doesn't shrink the weights — it shrinks the per-user cache. On 2xH100 (20 GB
headroom after a 70B FP16 load), at 32K context per user: BF16 fits ~2 users, TurboQuant fits ~11.


In [ ]:
headroom_gb = 20.0
per_user_bf16 = 10.0    # GB, 70B @ 32K, BF16
users_bf16 = int(headroom_gb // per_user_bf16)
users_tq   = int(headroom_gb // (per_user_bf16 / 6))
print(f"concurrent 32K users on 2xH100: {users_bf16} (BF16) -> {users_tq} (TurboQuant)")
assert users_tq >= 5 * users_bf16
print("OK: ~5.5x more users per GPU-hour — cost per user-hour falls ~5.5x with it")

### Takeaway

- The KV cache is the per-request tax on every long-context workload; at 128K a 70B model pays ~40 GB.
- PolarQuant's random rotation is the whole trick: it turns outlier-ridden vectors into near-Gaussian
  ones, where a trivial scalar quantizer is suddenly accurate — no calibration data, no learned parameters.
- The QJL residual sketch keeps the estimator unbiased, which is what attention needs.
- Honest caveat from independent reproductions: 3-bit is pristine on retrieval benchmarks (NIAH, LongBench),
  but reasoning benchmarks show 15–25 point drops — validate against your own eval set before trusting the headline.
